# Multi-cell-line PDAC dataset assembly and canonical unit construction

**Research context:** Master's thesis, *Machine Learning for Drug-Combination Modelling in Pancreatic Ductal Adenocarcinoma* (University of Helsinki, 2026).

This notebook consolidates the two original data-preparation notebooks supplied by the author:

1. `00_multicell_pdac_data_assembly` — load and harmonise the Jaaks/GDSC drug-combination screen, map the PDAC cell lines to DepMap, enrich drug identifiers using DrugComb, and integrate DepMap expression and pathway summaries.
2. `01_multicell_pdac_canonical_unit_construction` — canonicalise drug-pair order, aggregate concentration-level responses, enforce strict endpoints, correct the two documented SMILES entries, and generate Morgan fingerprints.

**Frozen dataset contract:** 53,441 concentration-level observations, 26 PDAC cell lines, 325 unique unordered drug pairs, and 2,483 positive concentration-level labels; the final dataset contains 8,450 canonical units, 1,063 any-positive units, 188 strict majority-positive units, and 999 mixed-label units.

**Important:** This notebook contains **code only**, not redistributed Jaaks, DrugComb or DepMap data. Supply the original input files locally in `data/raw/` and confirm their applicable licence and release. Both large final CSVs are saved locally to the `tables/` directory and are listed in `.gitignore`. The intermediate data generated under `data/processed/dataset_assembly/` must also stay outside public Git tracking.

**Run from the cloned repository root or from `notebooks/`, top to bottom, using a fresh Python kernel.** The local directory layout, source names, software dependencies and input contracts are described in [the data-access guide](../docs/DATA_ACCESS_AND_PREPROCESSING.md). Full execution requires the original releases and cannot be verified from an HTML export alone.

**Feature leakage note:** The original assembly creates descriptive 20-component expression PCA scores for within-cell-line drug-pair ranking. The *unseen-cell-line* validation notebook must fit its expression PCA **within the training fold** using the separate full-expression table generated here. The global PCs created in Stage A are not appropriate substitutes for cold-cell-line validation.


## Local dependencies and raw source layout

Install Python packages compatible with your environment: `numpy`, `pandas`, `scikit-learn`, `rdkit`, `openpyxl`, `ipython`, `jupyter`, and `nbformat` (the notebook itself uses IPython `display`).

Supply the original downloaded files under the following **local-only** directories before executing Stage A:

```text
data/raw/
├── Jaaks_GDSC_combinations/  # Jaaks original screen response CSV
├── DepMap/                   # Model.csv and Public 26Q1 expression CSV
└── DrugComb/                 # DrugComb_drug_identifiers.xlsx, sheet 'drug'
```

The source-specific allowed filenames and fallback patterns are defined in the next input-discovery cell; avoid placing ambiguous multiple versions in the same folder. All source files should retain their original provenance, checksum and licensing records. No download or licence acceptance is performed by this notebook.


## Stage A — Assemble the Jaaks/GDSC–DepMap concentration-level dataset


### Imports, project-root detection and paths


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
N_EXPRESSION_PCS = 20

#Manual override. Leave as None when the notebook is inside the repository.
PROJECT_ROOT_OVERRIDE = None

def detect_project_root(start: Path = Path.cwd()) -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
        return root

    start = start.expanduser().resolve()
    candidates = [start, *start.parents]

    for p in candidates:
        if (p / ".git").exists():
            return p

    for p in candidates:
        if (p / "data" / "raw").is_dir() and (p / "data" / "processed").exists():
            return p

    for p in candidates:
        if (p / "data" / "raw").is_dir():
            return p

    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside the "
        "repository or set PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = detect_project_root()

# Source data and generated datasets remain local; never add the generated
# data/processed/dataset_assembly/ directory to the public Git repository.

RAW_DIR = PROJECT_ROOT / "data" / "raw"
JAAKS_DIR = RAW_DIR / "Jaaks_GDSC_combinations"
DEPMAP_DIR = RAW_DIR / "DepMap"
DRUGCOMBDB_DIR = RAW_DIR / "DrugComb"

ASSEMBLY_DIR = PROJECT_ROOT / "data" / "processed" / "dataset_assembly"
AUDIT_DIR = PROJECT_ROOT / "results" / "tables" / "dataset_assembly"
ASSEMBLY_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

def rel(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Project root:", PROJECT_ROOT.name)
print("Assembly directory:", rel(ASSEMBLY_DIR))


### Reusable helpers


In [ ]:
def normalize_name(value: object) -> str | float:
    if pd.isna(value):
        return np.nan
    text = re.sub(r"[^A-Z0-9]", "", str(value).upper().strip())
    return np.nan if text in {"", "NAN", "NONE", "NULL"} else text


def clean_gene_symbol(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"\s+\(\d+\)$", "", text)
    return text.upper()


def first_existing_file(directory: Path, candidates: list[str], glob_patterns: list[str] | None = None) -> Path:
    directory = Path(directory)
    for name in candidates:
        path = directory / name
        if path.exists():
            return path
    for pattern in (glob_patterns or []):
        matches = sorted(directory.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not locate a required file in {directory}. "
        f"Tried {candidates} and patterns {glob_patterns or []}."
    )


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    sep = "\t" if path.suffix.lower() in {".tsv", ".txt"} else ","
    for encoding in ["utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, sep=sep, encoding="latin1", low_memory=False)


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def first_nonmissing(series: pd.Series):
    for value in series:
        if pd.notna(value) and str(value).strip() not in {"", "nan", "None"}:
            return value
    return np.nan


def make_pair_key(a: object, b: object) -> str:
    aa, bb = normalize_name(a), normalize_name(b)
    if pd.isna(aa) or pd.isna(bb):
        return np.nan
    return "||".join(sorted([str(aa), str(bb)]))


def parse_binary_synergy(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype("Int64")
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all() and set(numeric.unique()).issubset({0, 1}):
        return numeric.astype("Int64")
    mapped = (
        series.astype(str).str.strip().str.lower()
        .map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    )
    return mapped.astype("Int64")


def software_table() -> pd.DataFrame:
    return pd.DataFrame([
        {"package": "Python", "version": sys.version.split()[0]},
        {"package": "pandas", "version": pd.__version__},
        {"package": "NumPy", "version": np.__version__},
        {"package": "scikit-learn", "version": sklearn.__version__},
    ])


### Locate the raw thesis inputs


In [ ]:
JAAKS_PATH = first_existing_file(
    JAAKS_DIR,
    [
        "original_screen_drug_combination_data.csv",
        "jaaks_original_screen.csv",
        "gdsc_combinations_screen.csv",
    ],
    ["*original*screen*drug*combination*.csv", "*jaaks*.csv"],
)

DEPMAP_MODEL_PATH = first_existing_file(
    DEPMAP_DIR,
    ["Model.csv", "sample_info.csv"],
    ["*Model*.csv"],
)

DEPMAP_EXPRESSION_PATH = first_existing_file(
    DEPMAP_DIR,
    [
        "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
        "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
    ],
    ["*OmicsExpression*TPM*Logp1*.csv", "*Expression*Protein*Coding*Genes*.csv"],
)

DRUG_IDENTIFIER_PATH = first_existing_file(
    DRUGCOMBDB_DIR,
    ["DrugComb_drug_identifiers.xlsx"],
    ["*drug*identifier*.xlsx", "*Drug*Identifier*.xlsx"],
)

input_paths = {
    "Jaaks/GDSC response": JAAKS_PATH,
    "DepMap model metadata": DEPMAP_MODEL_PATH,
    "DepMap expression": DEPMAP_EXPRESSION_PATH,
    "DrugComb drug identifiers": DRUG_IDENTIFIER_PATH,
}

input_manifest = pd.DataFrame([
    {
        "artifact": name,
        "path_relative_to_repo": rel(path),
        "size_mb": path.stat().st_size / (1024 ** 2),
        "sha256": sha256_file(path),
    }
    for name, path in input_paths.items()
])
display(input_manifest)
input_manifest.to_csv(AUDIT_DIR / "multicell_data_assembly_input_manifest.csv", index=False)


### Load and filter the Jaaks/GDSC response screen


In [ ]:
jaaks_raw = read_table(JAAKS_PATH)
print("Raw Jaaks/GDSC shape:", jaaks_raw.shape)

if "Tissue" not in jaaks_raw.columns:
    raise KeyError("Expected a 'Tissue' column in the Jaaks/GDSC source table.")

jaaks = jaaks_raw[
    jaaks_raw["Tissue"].astype(str).str.strip().str.lower().eq("pancreas")
].copy()

if "Synergy" not in jaaks.columns:
    raise KeyError("Expected a 'Synergy' column in the Jaaks/GDSC source table.")

jaaks["label_synergy"] = parse_binary_synergy(jaaks["Synergy"])
if jaaks["label_synergy"].isna().any():
    raise ValueError("Could not parse all Jaaks/GDSC Synergy values as binary labels.")

required_drug_cols = ["ANCHOR_NAME", "LIBRARY_NAME"]
missing = [c for c in required_drug_cols if c not in jaaks.columns]
if missing:
    raise KeyError(f"Missing Jaaks/GDSC drug columns: {missing}")

jaaks["drug1_name_for_merge"] = jaaks["ANCHOR_NAME"]
jaaks["drug2_name_for_merge"] = jaaks["LIBRARY_NAME"]
jaaks["drug1_norm"] = jaaks["drug1_name_for_merge"].map(normalize_name)
jaaks["drug2_norm"] = jaaks["drug2_name_for_merge"].map(normalize_name)
jaaks["drug_pair_group"] = [
    make_pair_key(a, b)
    for a, b in zip(jaaks["drug1_norm"], jaaks["drug2_norm"])
]

print("PDAC/pancreas rows:", len(jaaks))
print("Positive rows:", int(jaaks["label_synergy"].sum()))
print("Row-level positive prevalence:", float(jaaks["label_synergy"].mean()))
print("Unique unordered drug pairs:", jaaks["drug_pair_group"].nunique())


### Map Jaaks/GDSC cell lines to DepMap `ModelID`


In [ ]:
depmap_model = read_table(DEPMAP_MODEL_PATH)

if "ModelID" not in depmap_model.columns:
    raise KeyError("DepMap model metadata must contain ModelID.")

mapped = False
if "SIDM" in jaaks.columns and "SangerModelID" in depmap_model.columns:
    sidm_lookup = (
        depmap_model[["ModelID", "SangerModelID"]]
        .dropna()
        .assign(SIDM=lambda d: d["SangerModelID"].astype(str))
        .drop_duplicates("SIDM")
    )
    jaaks = jaaks.merge(sidm_lookup[["SIDM", "ModelID"]], on="SIDM", how="left")
    mapped = True

if not mapped or jaaks["ModelID"].isna().any():
    name_lookup_parts = []
    for col in ["ModelName", "StrippedCellLineName", "CCLEName"]:
        if col in depmap_model.columns:
            temp = depmap_model[["ModelID", col]].dropna().copy()
            temp["cell_line_norm_lookup"] = temp[col].map(normalize_name)
            name_lookup_parts.append(temp[["cell_line_norm_lookup", "ModelID"]])

    if name_lookup_parts:
        name_lookup = (
            pd.concat(name_lookup_parts, ignore_index=True)
            .drop_duplicates("cell_line_norm_lookup")
        )
        jaaks["cell_line_norm_lookup"] = jaaks["CELL_LINE_NAME"].map(normalize_name)
        jaaks = jaaks.merge(
            name_lookup.rename(columns={"ModelID": "ModelID_name_match"}),
            on="cell_line_norm_lookup",
            how="left",
        )
        if "ModelID" not in jaaks.columns:
            jaaks["ModelID"] = jaaks["ModelID_name_match"]
        else:
            jaaks["ModelID"] = jaaks["ModelID"].combine_first(jaaks["ModelID_name_match"])
        jaaks = jaaks.drop(columns=["ModelID_name_match"], errors="ignore")

jaaks["ModelID"] = jaaks["ModelID"].replace({"nan": np.nan, "None": np.nan})
print("Rows with ModelID:", int(jaaks["ModelID"].notna().sum()), "/", len(jaaks))
print("Mapped cell lines:", jaaks.loc[jaaks["ModelID"].notna(), "ModelID"].nunique())


### Build a compact Jaaks drug master and enrich it with DrugComb identifiers


In [ ]:
drug_rows = []

role_specs = [
    ("ANCHOR", "ANCHOR_NAME"),
    ("LIBRARY", "LIBRARY_NAME"),
]

for prefix, name_col in role_specs:
    if name_col not in jaaks.columns:
        continue
    temp = pd.DataFrame({
        "drug_name": jaaks[name_col],
        "drug_norm": jaaks[name_col].map(normalize_name),
        "target": jaaks.get(f"{prefix}_TARGET", pd.Series(np.nan, index=jaaks.index)),
        "pathway": jaaks.get(f"{prefix}_PATHWAY", pd.Series(np.nan, index=jaaks.index)),
        "drug_type": jaaks.get(f"{prefix}_DRUG_TYPE", pd.Series(np.nan, index=jaaks.index)),
        "clinical_relevance": jaaks.get(f"{prefix}_Clin_Rel", pd.Series(np.nan, index=jaaks.index)),
    })
    drug_rows.append(temp)

drug_long = pd.concat(drug_rows, ignore_index=True).dropna(subset=["drug_norm"])
drug_master = (
    drug_long.groupby("drug_norm", as_index=False)
    .agg(
        drug_name=("drug_name", first_nonmissing),
        target=("target", first_nonmissing),
        pathway=("pathway", first_nonmissing),
        drug_type=("drug_type", first_nonmissing),
        clinical_relevance=("clinical_relevance", first_nonmissing),
    )
)

drug_ids_raw = pd.read_excel(DRUG_IDENTIFIER_PATH, sheet_name="drug")
drug_ids_raw = drug_ids_raw.replace(
    ["NA", "NaN", "nan", "\\N", "", "None", "NONE", "NULL"],
    np.nan,
)

if "dname" not in drug_ids_raw.columns:
    raise KeyError("DrugComb drug identifier table must contain 'dname'.")

identifier = drug_ids_raw.copy()
identifier["identifier_norm"] = identifier["dname"].map(normalize_name)

rename_map = {
    "id": "drugcomb_identifier_id",
    "cid": "pubchem_cid",
    "clinical_phase": "clinical_phase_drugcomb",
    "synonyms": "drugcomb_synonyms",
    "target_name": "drugcomb_target_name_raw",
    "target_type": "drugcomb_target_type_raw",
}
identifier = identifier.rename(columns={k: v for k, v in rename_map.items() if k in identifier.columns})

identifier_keep = [
    "identifier_norm", "dname", "smiles", "pubchem_cid", "chembl_id", "inchikey",
    "molecular_formula", "clinical_phase_drugcomb", "drugbank_id", "kegg_id",
    "drugcomb_identifier_id", "drugcomb_synonyms",
    "drugcomb_target_name_raw", "drugcomb_target_type_raw", "stitch_name",
]
identifier = identifier[[c for c in identifier_keep if c in identifier.columns]]

# Keep the uncollapsed identifier rows for synonym/alias discovery. DrugComb may
# contain more than one record for the same canonical drug name, so these rows
# must NOT be merged directly into the Jaaks drug master or they can duplicate
# response observations.
identifier_for_aliases = identifier.copy()

alias_rows = []
for row in identifier_for_aliases.itertuples(index=False):
    record = row._asdict()
    canonical = record.get("identifier_norm")
    if pd.isna(canonical):
        continue
    aliases = [record.get("dname"), record.get("stitch_name")]
    synonyms = record.get("drugcomb_synonyms")
    if pd.notna(synonyms):
        aliases.extend(str(synonyms).split(";"))
    for alias in aliases:
        alias_norm = normalize_name(alias)
        if pd.notna(alias_norm):
            alias_rows.append({
                "alias_norm": alias_norm,
                "identifier_norm": canonical,
            })

alias_lookup = (
    pd.DataFrame(alias_rows)
    .drop_duplicates("alias_norm")
)

# Collapse DrugComb to exactly one metadata row per canonical identifier before
# attaching it to the Jaaks drug master. This prevents a many-to-many metadata
# merge from multiplying experimental response rows.
identifier = identifier.dropna(subset=["identifier_norm"]).copy()
identifier_agg = {
    col: first_nonmissing
    for col in identifier.columns
    if col != "identifier_norm"
}
identifier = (
    identifier.groupby("identifier_norm", as_index=False)
    .agg(identifier_agg)
)

drug_master = drug_master.merge(
    alias_lookup,
    left_on="drug_norm",
    right_on="alias_norm",
    how="left",
)
drug_master["identifier_norm"] = drug_master["identifier_norm"].fillna(drug_master["drug_norm"])
drug_master = drug_master.merge(
    identifier.add_suffix("_id"),
    left_on="identifier_norm",
    right_on="identifier_norm_id",
    how="left",
    validate="many_to_one",
)

for field in [
    "smiles", "pubchem_cid", "chembl_id", "inchikey", "molecular_formula",
    "clinical_phase_drugcomb", "drugbank_id", "kegg_id",
    "drugcomb_identifier_id", "drugcomb_synonyms",
    "drugcomb_target_name_raw", "drugcomb_target_type_raw",
]:
    source_col = f"{field}_id"
    if source_col in drug_master.columns:
        drug_master[field] = drug_master[source_col]

helper_cols = [
    c for c in drug_master.columns
    if c.endswith("_id") or c in {"alias_norm", "identifier_norm"}
]
drug_master = drug_master.drop(columns=helper_cols, errors="ignore")

if drug_master["drug_norm"].duplicated().any():
    dupes = drug_master.loc[
        drug_master["drug_norm"].duplicated(keep=False),
        ["drug_norm", "drug_name"],
    ].sort_values("drug_norm")
    raise AssertionError(
        "Drug metadata is not one-row-per-drug after harmonisation.\n"
        + dupes.to_string(index=False)
    )

print("Drug master rows:", len(drug_master))
print("Unique drug_norm values:", drug_master["drug_norm"].nunique())

drug_master.to_csv(ASSEMBLY_DIR / "drug_master_jaaks_with_drugcomb_identifiers.csv", index=False)

coverage = pd.DataFrame([
    {
        "field": field,
        "available": int(drug_master[field].notna().sum()),
        "total": len(drug_master),
        "coverage": float(drug_master[field].notna().mean()),
    }
    for field in [
        "smiles", "target", "pathway", "drug_type", "clinical_relevance",
        "clinical_phase_drugcomb", "pubchem_cid", "chembl_id", "inchikey",
    ]
    if field in drug_master.columns
])
display(coverage)


### Enrich the Jaaks/GDSC response rows with drug metadata


In [ ]:
drug_fields = [
    "drug_norm", "smiles", "pubchem_cid", "chembl_id", "inchikey",
    "molecular_formula", "clinical_phase_drugcomb", "drugbank_id", "kegg_id",
    "target", "pathway", "drug_type", "clinical_relevance",
    "drugcomb_target_name_raw", "drugcomb_target_type_raw",
]
drug_for_merge = drug_master[[c for c in drug_fields if c in drug_master.columns]].copy()

if drug_for_merge["drug_norm"].duplicated().any():
    raise AssertionError(
        "drug_for_merge must contain exactly one metadata row per normalized drug."
    )

enriched = jaaks.copy()
n_rows_before_drug_merge = len(enriched)

enriched = enriched.merge(
    drug_for_merge.add_prefix("drug1_"),
    left_on="drug1_norm",
    right_on="drug1_drug_norm",
    how="left",
    validate="many_to_one",
)
enriched = enriched.merge(
    drug_for_merge.add_prefix("drug2_"),
    left_on="drug2_norm",
    right_on="drug2_drug_norm",
    how="left",
    validate="many_to_one",
)

if len(enriched) != n_rows_before_drug_merge:
    raise AssertionError(
        "Drug metadata enrichment changed the number of experimental response rows: "
        f"{n_rows_before_drug_merge:,} -> {len(enriched):,}. "
        "This indicates a non-unique metadata merge key."
    )

for role in ["drug1", "drug2"]:
    duplicate_norm = f"{role}_drug_norm"
    if duplicate_norm in enriched.columns:
        enriched = enriched.drop(columns=[duplicate_norm])

required_structure_cols = ["drug1_smiles", "drug2_smiles"]
missing_structure_cols = [c for c in required_structure_cols if c not in enriched.columns]
if missing_structure_cols:
    raise KeyError(f"Missing enriched structure columns: {missing_structure_cols}")

print("Enriched PDAC rows:", enriched.shape)
print(
    "Both-drug SMILES coverage:",
    enriched[required_structure_cols].notna().all(axis=1).mean(),
)


### Load DepMap expression and construct the 26-cell-line expression representation


In [ ]:
expr_raw = read_table(DEPMAP_EXPRESSION_PATH)

# Make ModelID a column if the file was read with it as the index-like first field.
if "ModelID" not in expr_raw.columns:
    first_col = expr_raw.columns[0]
    if str(first_col).lower() in {"modelid", "index", "unnamed: 0"}:
        expr_raw = expr_raw.rename(columns={first_col: "ModelID"})
    else:
        raise KeyError("DepMap expression table must contain ModelID.")

for default_col in ["IsDefaultEntryForModel", "IsDefaultEntryForMC", "is_default_entry"]:
    if default_col in expr_raw.columns:
        mask = expr_raw[default_col].astype(str).str.upper().isin(["YES", "TRUE", "1"])
        if mask.any():
            expr_raw = expr_raw.loc[mask].copy()
        break

gene_cols_raw = [
    c for c in expr_raw.columns
    if re.search(r"\s+\(\d+\)$", str(c))
]
if not gene_cols_raw:
    metadata_names = {
        "ModelID", "SequencingID", "ModelConditionID",
        "IsDefaultEntryForMC", "IsDefaultEntryForModel",
    }
    gene_cols_raw = [
        c for c in expr_raw.columns
        if c not in metadata_names
        and pd.to_numeric(expr_raw[c], errors="coerce").notna().mean() > 0.95
    ]

if not gene_cols_raw:
    raise ValueError("Could not identify DepMap gene-expression columns.")

expression = expr_raw[["ModelID"] + gene_cols_raw].copy()
for col in gene_cols_raw:
    expression[col] = pd.to_numeric(expression[col], errors="coerce")

expression = expression.rename(columns={c: clean_gene_symbol(c) for c in gene_cols_raw})
expression = expression.set_index("ModelID").T.groupby(level=0).mean().T
expression = expression.groupby(level=0).mean()
expression.index = expression.index.astype(str)

jaaks_model_ids = enriched["ModelID"].dropna().astype(str).unique()
matched_ids = expression.index.intersection(jaaks_model_ids)

# Retain the matched 26-cell-line expression matrix for downstream training-fold PCA. 
# The current assembly contains 19,215 gene columns.
# When saved as CSV, ModelID adds one identifier column, giving 19,216 columns in total.
# The descriptive global PCA below uses complete-case expression features; unseen-cell-line models must refit preprocessing per fold.

expression_for_fold_pca = expression.loc[matched_ids].copy().sort_index()
expression_for_fold_pca.index.name = "ModelID"

if expression_for_fold_pca.shape != (26, 19215):
    raise AssertionError(
        "Expected 26 PDAC cell lines × 19,215 gene-expression features; "
        f"found {expression_for_fold_pca.shape}. "
        "Check the DepMap release and gene harmonisation."
    )

print(
    "Verified expression matrix:",
    expression_for_fold_pca.shape,
    "(26 cell lines × 19,215 genes)"
)

expression_matched = expression.loc[matched_ids].copy()
expression_matched = expression_matched.dropna(axis=1, how="any")

print("Matched Jaaks/GDSC cell lines with complete DepMap expression:", len(expression_matched))
print("Expression genes retained:", expression_matched.shape[1])

if len(expression_matched) < 2:
    raise ValueError("Too few expression-matched cell lines for PCA.")

# Fit PCA only on the cell lines actually represented in the final expression-complete table.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(expression_matched)

n_components = min(N_EXPRESSION_PCS, len(expression_matched) - 1)
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_cols = [f"expr_pca_{i+1}" for i in range(n_components)]
expression_pca = pd.DataFrame(
    X_pca,
    index=expression_matched.index,
    columns=pca_cols,
).reset_index().rename(columns={"index": "ModelID"})

pca_variance = pd.DataFrame({
    "component": pca_cols,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
})
pca_variance.to_csv(
    ASSEMBLY_DIR / "depmap_jaaks_matched_expression_pca_variance_report.csv",
    index=False,
)

# Compact pathway scores retained as supporting molecular-context features.
GENE_SETS = {
    "cell_cycle": ["MKI67","TOP2A","CCNA2","CCNB1","CCNB2","CDK1","CDK2","CDC20","CDC25C","AURKA","AURKB"],
    "dna_replication_genome_integrity": ["MCM2","MCM3","MCM4","MCM5","MCM6","MCM7","PCNA","CHEK1","CHEK2","ATR","ATM","BRCA1","BRCA2","RAD51","PARP1","TOP1"],
    "apoptosis": ["BAX","BAK1","BCL2","BCL2L1","MCL1","CASP3","CASP7","CASP8","CASP9","XIAP","BIRC5"],
    "mapk_signaling": ["KRAS","NRAS","BRAF","RAF1","MAP2K1","MAP2K2","MAPK1","MAPK3","DUSP6","SPRY2","ETV4","ETV5"],
    "pi3k_mtor": ["PIK3CA","PIK3CB","PIK3R1","AKT1","AKT2","MTOR","RPTOR","RICTOR","TSC1","TSC2","PTEN"],
    "emt_tgf_beta": ["VIM","ZEB1","ZEB2","SNAI1","SNAI2","TWIST1","TGFB1","TGFBR1","TGFBR2","SERPINE1","FN1","CDH2"],
    "classical_pancreatic_identity": ["GATA6","HNF1A","HNF4A","FOXA2","PDX1","EPCAM","KRT8","KRT18","KRT19"],
    "basal_squamous": ["KRT5","KRT6A","KRT14","KRT17","S100A2","TP63","LAMC2"],
}

pathway_scores = pd.DataFrame(index=expression_matched.index)
pathway_coverage_rows = []
for name, genes in GENE_SETS.items():
    present = [g for g in genes if g in expression_matched.columns]
    pathway_scores[name] = expression_matched[present].mean(axis=1) if present else np.nan
    pathway_coverage_rows.append({
        "pathway": name,
        "n_genes_total": len(genes),
        "n_genes_present": len(present),
        "present_genes": ", ".join(present),
    })

pathway_scores = pathway_scores.reset_index().rename(columns={"index": "ModelID"})
pd.DataFrame(pathway_coverage_rows).to_csv(
    ASSEMBLY_DIR / "depmap_jaaks_matched_pathway_coverage.csv",
    index=False,
)


### Create the final expression-complete concentration-level modelling table


In [ ]:
# Keep only rows whose cell line has the expression representation fitted above.
final_rows = enriched[
    enriched["ModelID"].astype(str).isin(set(expression_matched.index.astype(str)))
].copy()

final_rows = final_rows.merge(expression_pca, on="ModelID", how="left", validate="many_to_one")
final_rows = final_rows.merge(pathway_scores, on="ModelID", how="left", validate="many_to_one")

# Recompute the unordered pair key after all merges.
final_rows["drug_pair_group"] = [
    make_pair_key(a, b)
    for a, b in zip(final_rows["drug1_norm"], final_rows["drug2_norm"])
]

# Preserve a full enriched version before the expression-complete filter as a provenance artifact.
enriched.to_csv(
    ASSEMBLY_DIR / "jaaks_pdac_enriched_all_rows.csv",
    index=False,
)

FINAL_SOURCE_PATH = ASSEMBLY_DIR / "final_ml_ready_jaaks_pdac_enriched_with_expression_pca.csv"
final_rows.to_csv(FINAL_SOURCE_PATH, index=False)

print("Final concentration-level table:", final_rows.shape)
print("Cell lines:", final_rows["ModelID"].nunique())
print("Unordered drug pairs:", final_rows["drug_pair_group"].nunique())
print("Positive rows:", int(final_rows["label_synergy"].sum()))
print("Positive prevalence:", float(final_rows["label_synergy"].mean()))
print("Output:", rel(FINAL_SOURCE_PATH))


### Thesis-contract audit and reproducibility records


In [ ]:
# The hard assertions below intentionally encode the frozen thesis dataset.
# If a source release or harmonisation rule changes, stop and investigate rather than
# silently producing a different training population.

expected = {
    "concentration_rows": 53441,
    "cell_lines": 26,
    "unordered_drug_pairs": 325,
    "positive_rows": 2483,
}

observed = {
    "concentration_rows": len(final_rows),
    "cell_lines": int(final_rows["ModelID"].nunique()),
    "unordered_drug_pairs": int(final_rows["drug_pair_group"].nunique()),
    "positive_rows": int(final_rows["label_synergy"].sum()),
}

contract = pd.DataFrame([
    {
        "quantity": key,
        "expected": expected[key],
        "observed": observed[key],
        "pass": expected[key] == observed[key],
    }
    for key in expected
])
display(contract)

if not contract["pass"].all():
    raise AssertionError(
        "The assembled concentration-level dataset does not match the frozen thesis "
        "contract. Inspect input versions, identifier mapping and expression filtering."
    )

software_table().to_csv(AUDIT_DIR / "multicell_data_assembly_software_versions.csv", index=False)

output_manifest = pd.DataFrame([
    {
        "artifact": "final concentration-level modelling table",
        "path_relative_to_repo": rel(FINAL_SOURCE_PATH),
        "size_mb": FINAL_SOURCE_PATH.stat().st_size / (1024 ** 2),
        "sha256": sha256_file(FINAL_SOURCE_PATH),
    },
    {
        "artifact": "drug master",
        "path_relative_to_repo": rel(ASSEMBLY_DIR / "drug_master_jaaks_with_drugcomb_identifiers.csv"),
        "size_mb": (ASSEMBLY_DIR / "drug_master_jaaks_with_drugcomb_identifiers.csv").stat().st_size / (1024 ** 2),
        "sha256": sha256_file(ASSEMBLY_DIR / "drug_master_jaaks_with_drugcomb_identifiers.csv"),
    },
])
output_manifest.to_csv(AUDIT_DIR / "multicell_data_assembly_output_manifest.csv", index=False)
display(output_manifest)

dataset_card = f"""# Multi-cell-line PDAC concentration-level dataset card

## Provenance
- Supervised response source: Jaaks/GDSC pancreatic drug-combination screen.
- Cell-line molecular context: DepMap/CCLE expression.
- Drug identifiers / SMILES: DrugComb identifier table.
- DrugComb response measurements are not used as supervised labels here.
- TCGA-PAAD is not part of this supervised response dataset.

## Frozen modelling source
- Rows: {len(final_rows):,}
- PDAC cell lines: {final_rows['ModelID'].nunique():,}
- Unordered drug pairs: {final_rows['drug_pair_group'].nunique():,}
- Row-level positives: {int(final_rows['label_synergy'].sum()):,}
- Row-level positive prevalence: {final_rows['label_synergy'].mean():.4f}
- Expression PCs: {len(pca_cols)}
- Cumulative variance explained by retained PCs: {pca_variance['cumulative_variance'].iloc[-1]:.4f}

## Feature-policy note
Clinical-relevance text is retained as descriptive metadata but is not part of the final
supervised annotation feature block.

Notebook 01 collapses repeated concentration rows to strict canonical biological units.
"""
(AUDIT_DIR / "multicell_concentration_dataset_card.md").write_text(
    dataset_card,
    encoding="utf-8",
)
print(dataset_card)


### Stage A output and checkpoint

The validated intermediate table is written locally to:

`data/processed/dataset_assembly/final_ml_ready_jaaks_pdac_enriched_with_expression_pca.csv`

Stage B reads exactly this Stage A CSV to preserve the original processing boundary, and derives order-invariant per-pair model units. Stage A retains expression_for_fold_pca in memory for downstream grouped validation. The validated matrix contains 26 PDAC cell lines and 19,215 gene-expression features. When saved as a CSV, the additional ModelID identifier produces 19,216 total columns. The original PCA preprocessing is retained, while downstream unseen-cell-line validation must refit expression preprocessing within each training fold.


## Stage B — Canonical unit construction, endpoint audit and Morgan fingerprints


### Stage B configuration and output locations

This stage reproduces the aggregation and Morgan calculation from the original notebook 01. The final local outputs include both `tables/pdac_multicell_drug_pair_units_with_morgan.csv` (required by grouped validation and ranking) and `tables/depmap_pdac_full_expression_for_fold_pca.csv` (required for training-fold PCA). The intermediate pickle and audit reports are preserved; all such outputs must remain **untracked** on the public repository.


In [ ]:
# Original notebook 01 adds RDKit for documented SMILES corrections and Morgan bits.
import shutil
from rdkit import Chem, DataStructs, rdBase, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.error")

MORGAN_BITS = 512
MORGAN_RADIUS = 2
WRITE_LEGACY_COMPATIBILITY_COPY = False  # True if an old local script needs it

SOURCE_PATH = FINAL_SOURCE_PATH   # generated and contract-checked in Stage A
AUDIT_DIR = (PROJECT_ROOT / "results" / "tables" / "ml_comprehensive"
             / "canonical_unit_dataset")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

AUTHORITATIVE_PICKLE = ASSEMBLY_DIR / "canonical_order_invariant_unit_with_morgan.pkl"
AUTHORITATIVE_CSV = ASSEMBLY_DIR / "canonical_pdac_units_strict_no_fingerprints.csv"

TABLE_DIR = PROJECT_ROOT / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_MODELLING_CSV = TABLE_DIR / "pdac_multicell_drug_pair_units_with_morgan.csv"
EXPRESSION_FOLD_CSV = TABLE_DIR / "depmap_pdac_full_expression_for_fold_pca.csv"

LEGACY_COMPAT_DIR = (PROJECT_ROOT / "results" / "tables"
                     / "ml_comprehensive" / "final_three_notebook_workflow")
LEGACY_COMPAT_PATH = LEGACY_COMPAT_DIR / "canonical_order_invariant_unit_with_morgan.pkl"

print("RDKit:", rdBase.rdkitVersion)
print("Canonical input:", rel(SOURCE_PATH))
print("Canonical local output:", rel(CANONICAL_MODELLING_CSV))
print("Fold-wise expression local output:", rel(EXPRESSION_FOLD_CSV))


### Helpers


In [ ]:
def normalize_drug_key(value: object) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def make_pair_key(a: object, b: object) -> str:
    aa, bb = normalize_drug_key(a), normalize_drug_key(b)
    return "||".join(sorted([aa, bb]))


def pretty_pair_key(key: object) -> str:
    return str(key).replace("||", " + ")


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def valid_smiles(value: object) -> bool:
    if pd.isna(value):
        return False
    try:
        return Chem.MolFromSmiles(str(value)) is not None
    except Exception:
        return False


SMILES_OVERRIDES = {
    # Frozen corrections carried forward from the earlier structure audit.
    # OSI-027 had a malformed source structure with an extra separator.
    "OSI027": "COC1=CC=CC2=CC(=C3C4=C(N=CNN4C(=N3)C5CCC(CC5)C(=O)O)N)N=C21",
    # OXALIPLATIN was missing from the DrugComb identifier export used in notebook 00.
    # This is the structure used by the frozen 26-drug corrected SMILES master.
    "OXALIPLATIN": "NC1CCCCC1N.O=C([O-])C(=O)[O-].[Pt+2]",
}


def clean_and_canonicalize_smiles(smiles: object, drug_key: object) -> str | float:
    key = normalize_drug_key(drug_key)

    # Apply documented drug-specific corrections first. This must happen before
    # the missing-value return because OXALIPLATIN is absent from the DrugComb
    # identifier table but has a frozen project structure.
    if key in SMILES_OVERRIDES:
        value = SMILES_OVERRIDES[key]
    else:
        if pd.isna(smiles):
            return np.nan

        value = str(smiles).replace("\u200b", "").replace("\ufeff", "").strip()
        if value.lower() in {"", "nan", "none", "null"}:
            return np.nan

        # Remove a terminal semicolon / extra structure separator if present.
        value = re.split(r"[;；]", value, maxsplit=1)[0].strip()

    mol = Chem.MolFromSmiles(value)
    if mol is None:
        return np.nan

    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


### Load the concentration-level table and record provenance


In [ ]:
source = pd.read_csv(SOURCE_PATH, low_memory=False)

required = {
    "ModelID", "label_synergy", "drug1_norm", "drug2_norm",
    "drug1_smiles", "drug2_smiles",
}
missing = required - set(source.columns)
if missing:
    raise KeyError(f"Missing required source columns: {sorted(missing)}")

source["label_synergy"] = pd.to_numeric(source["label_synergy"], errors="coerce")
source = source.dropna(
    subset=["ModelID", "drug1_norm", "drug2_norm", "label_synergy"]
).copy()
source["label_synergy"] = source["label_synergy"].astype(int)

if not source["label_synergy"].isin([0, 1]).all():
    raise ValueError("label_synergy must be binary.")

source["drug_pair_group"] = [
    make_pair_key(a, b)
    for a, b in zip(source["drug1_norm"], source["drug2_norm"])
]

software_versions = pd.DataFrame([
    {"package": "Python", "version": sys.version.split()[0]},
    {"package": "pandas", "version": pd.__version__},
    {"package": "NumPy", "version": np.__version__},
    {"package": "scikit-learn", "version": sklearn.__version__},
    {"package": "RDKit", "version": rdBase.rdkitVersion},
])
software_versions.to_csv(AUDIT_DIR / "canonical_unit_software_versions.csv", index=False)

input_manifest = pd.DataFrame([{
    "artifact": "concentration-level modelling table",
    "path_relative_to_repo": rel(SOURCE_PATH),
    "size_mb": SOURCE_PATH.stat().st_size / (1024 ** 2),
    "sha256": sha256_file(SOURCE_PATH),
}])
input_manifest.to_csv(AUDIT_DIR / "canonical_unit_input_manifest.csv", index=False)

source_overview = pd.DataFrame([
    {"quantity": "concentration-level rows", "value": len(source)},
    {"quantity": "PDAC cell lines", "value": source["ModelID"].nunique()},
    {"quantity": "unordered drug pairs", "value": source["drug_pair_group"].nunique()},
    {"quantity": "positive rows", "value": int(source["label_synergy"].sum())},
    {"quantity": "row-level positive prevalence", "value": float(source["label_synergy"].mean())},
])
display(source_overview)


### Audit repeated measurements and response fields


In [ ]:
unit_keys = ["ModelID", "drug_pair_group"]

repeat_audit = (
    source.groupby(unit_keys)
    .agg(
        n_rows=("label_synergy", "size"),
        n_positive=("label_synergy", "sum"),
        n_label_values=("label_synergy", "nunique"),
    )
    .reset_index()
)
repeat_audit["positive_fraction"] = repeat_audit["n_positive"] / repeat_audit["n_rows"]
repeat_audit["has_conflicting_labels"] = repeat_audit["n_label_values"].gt(1)

repeat_summary = pd.DataFrame([
    {"quantity": "concentration-level rows", "value": len(source)},
    {"quantity": "canonical ModelID-pair units", "value": len(repeat_audit)},
    {"quantity": "units with >1 concentration row", "value": int((repeat_audit["n_rows"] > 1).sum())},
    {"quantity": "units with conflicting labels", "value": int(repeat_audit["has_conflicting_labels"].sum())},
    {"quantity": "median rows per unit", "value": float(repeat_audit["n_rows"].median())},
    {"quantity": "maximum rows per unit", "value": int(repeat_audit["n_rows"].max())},
])
repeat_summary.to_csv(AUDIT_DIR / "unit_repetition_summary.csv", index=False)
repeat_audit.to_csv(AUDIT_DIR / "unit_label_audit.csv", index=False)
display(repeat_summary)

# Audit the old "Synergy" field and any other score-like columns so that a binary
# field is not accidentally reported as a genuine continuous assay outcome.
candidate_response_cols = [
    c for c in source.columns
    if re.search(r"synergy|score|bliss|loewe|hsa|zip|css|gamma", c, re.I)
]
response_rows = []
for column in candidate_response_cols:
    values = source[column]
    numeric = pd.to_numeric(values, errors="coerce")
    response_rows.append({
        "column": column,
        "dtype": str(values.dtype),
        "n_unique": int(values.nunique(dropna=True)),
        "missing_fraction": float(values.isna().mean()),
        "minimum": float(numeric.min()) if numeric.notna().any() else np.nan,
        "maximum": float(numeric.max()) if numeric.notna().any() else np.nan,
        "is_binary_numeric": (
            bool(set(numeric.dropna().unique()).issubset({0, 1}))
            if numeric.notna().any() else False
        ),
    })

response_audit = pd.DataFrame(response_rows).sort_values(
    ["n_unique", "column"], ascending=[False, True]
)
response_audit.to_csv(AUDIT_DIR / "continuous_response_candidate_audit.csv", index=False)
display(response_audit)


### Canonicalise drug order before aggregating units


In [ ]:
# Every paired drug attribute must move with its drug identity when the orientation is swapped.
PAIRED_SUFFIXES = [
    "smiles",
    "target",
    "pathway",
    "drug_type",
    "clinical_relevance",
    "clinical_phase_drugcomb",
    "drugcomb_target_name_raw",
    "drugcomb_target_type_raw",
    "pubchem_cid",
    "chembl_id",
    "inchikey",
    "molecular_formula",
    "drugbank_id",
    "kegg_id",
]

canonical_rows = source.copy()
key1 = canonical_rows["drug1_norm"].map(normalize_drug_key)
key2 = canonical_rows["drug2_norm"].map(normalize_drug_key)
swap = key1 > key2

canonical_rows["drug_a_norm"] = np.where(swap, key2, key1)
canonical_rows["drug_b_norm"] = np.where(swap, key1, key2)
canonical_rows["candidate_pair_key"] = [
    f"{a}||{b}"
    for a, b in zip(canonical_rows["drug_a_norm"], canonical_rows["drug_b_norm"])
]
canonical_rows["drug_pair_group"] = canonical_rows["candidate_pair_key"]

for suffix in PAIRED_SUFFIXES:
    left = f"drug1_{suffix}"
    right = f"drug2_{suffix}"
    if left in canonical_rows.columns and right in canonical_rows.columns:
        canonical_rows[f"drug_a_{suffix}"] = np.where(
            swap, canonical_rows[right], canonical_rows[left]
        )
        canonical_rows[f"drug_b_{suffix}"] = np.where(
            swap, canonical_rows[left], canonical_rows[right]
        )

# The molecular context belongs to the cell line, not to either drug.
pca_cols = sorted(
    [c for c in canonical_rows.columns if re.fullmatch(r"expr_pca_\d+", str(c))],
    key=lambda c: int(str(c).rsplit("_", 1)[-1]),
)

print("Expression PCA columns:", len(pca_cols))
print("Rows with orientation swap:", int(swap.sum()))


### Build strict canonical endpoints


In [ ]:
target_table = (
    canonical_rows.groupby(unit_keys, dropna=False)["label_synergy"]
    .agg(
        n_rows="size",
        n_positive="sum",
    )
    .reset_index()
)
target_table["n_negative"] = target_table["n_rows"] - target_table["n_positive"]
target_table["positive_fraction"] = (
    target_table["n_positive"] / target_table["n_rows"]
)

# FINAL ENDPOINT CONTRACT
target_table["any_positive_label"] = (
    target_table["positive_fraction"] > 0
).astype(int)

target_table["majority_label"] = (
    target_table["positive_fraction"] > 0.5
).astype(int)

target_table["has_conflicting_labels"] = target_table[
    "positive_fraction"
].between(0, 1, inclusive="neither")

# Audit the exact 0.5 boundary that caused the legacy >=0.5 definition to differ.
target_table["exact_half_positive"] = np.isclose(
    target_table["positive_fraction"].to_numpy(dtype=float),
    0.5,
)

endpoint_audit = pd.DataFrame([
    {
        "endpoint": "any_positive_label",
        "definition": "positive_fraction > 0",
        "n_units": len(target_table),
        "positive_count": int(target_table["any_positive_label"].sum()),
        "prevalence": float(target_table["any_positive_label"].mean()),
    },
    {
        "endpoint": "majority_label",
        "definition": "positive_fraction > 0.5",
        "n_units": len(target_table),
        "positive_count": int(target_table["majority_label"].sum()),
        "prevalence": float(target_table["majority_label"].mean()),
    },
])
endpoint_audit.to_csv(AUDIT_DIR / "strict_endpoint_audit.csv", index=False)
display(endpoint_audit)

print("Units exactly at positive_fraction = 0.5:", int(target_table["exact_half_positive"].sum()))


### Verify that unit-level features are constant and aggregate them


In [ ]:
canonical_feature_cols = [
    c for c in canonical_rows.columns
    if c.startswith("drug_a_") or c.startswith("drug_b_")
]
feature_cols = list(dict.fromkeys(canonical_feature_cols + pca_cols))

consistency_rows = []
for column in feature_cols:
    n_unique = (
        canonical_rows.groupby(unit_keys, dropna=False)[column]
        .nunique(dropna=True)
    )
    consistency_rows.append({
        "column": column,
        "inconsistent_units": int((n_unique > 1).sum()),
        "maximum_unique_values_within_unit": int(n_unique.max()) if len(n_unique) else 0,
    })

feature_consistency = (
    pd.DataFrame(consistency_rows)
    .sort_values(["inconsistent_units", "column"], ascending=[False, True])
)
feature_consistency.to_csv(
    AUDIT_DIR / "canonical_feature_consistency.csv",
    index=False,
)
display(feature_consistency.head(20))

core_cols = [
    "drug_a_norm", "drug_b_norm", "drug_a_smiles", "drug_b_smiles", *pca_cols
]
core_inconsistent = feature_consistency[
    feature_consistency["column"].isin(core_cols)
    & feature_consistency["inconsistent_units"].gt(0)
]
if len(core_inconsistent):
    raise AssertionError(
        "Core canonical features vary within a biological unit. Inspect the "
        "orientation or expression merge before proceeding."
    )

first_values = (
    canonical_rows.groupby(unit_keys, dropna=False)[feature_cols]
    .first()
    .reset_index()
)

unit = target_table.merge(
    first_values,
    on=unit_keys,
    how="left",
    validate="one_to_one",
)

unit["candidate_pair_key"] = unit["drug_pair_group"].astype(str)
unit["candidate_pair_label"] = unit["candidate_pair_key"].map(pretty_pair_key)
unit = unit.sort_values(["ModelID", "candidate_pair_key"]).reset_index(drop=True)

assert not unit.duplicated(["ModelID", "candidate_pair_key"]).any()

# Descriptive response information only — never a model predictor.
if "SYNERGY_DELTA_fAUC" in canonical_rows.columns:
    delta_fauc_by_unit = (
        canonical_rows.groupby(unit_keys, dropna=False)[
            "SYNERGY_DELTA_fAUC"
        ]
        .mean()
        .rename("delta_fauc_mean")
        .reset_index()
    )

    unit = unit.merge(
        delta_fauc_by_unit,
        on=unit_keys,
        how="left",
        validate="one_to_one",
    )

    assert unit["delta_fauc_mean"].notna().all()
    print("Canonical mean ΔfAUC available for", len(unit), "units")


### Clean/canonicalise SMILES and document structure corrections

The source identifier merge does not provide a valid structure for every drug. Two frozen,
documented corrections are applied here before fingerprint generation:

* **OSI-027**: replace the malformed source structure used in the early workflow;
* **OXALIPLATIN**: supply the structure from the project's frozen 26-drug corrected
  SMILES master because the DrugComb identifier export used in notebook 00 does not
  provide it.

All corrected structures are parsed and canonicalised with RDKit, and the before/after
status is written to the SMILES audit table.


In [ ]:
smiles_audit_parts = []

for role in ["a", "b"]:
    smiles_col = f"drug_{role}_smiles"
    drug_col = f"drug_{role}_norm"

    if smiles_col not in unit.columns:
        raise KeyError(f"Missing canonical SMILES column: {smiles_col}")

    original = unit[smiles_col].copy()
    cleaned = [
        clean_and_canonicalize_smiles(smiles, drug)
        for smiles, drug in zip(original, unit[drug_col])
    ]
    unit[smiles_col] = cleaned

    audit = pd.DataFrame({
        "role": role,
        "drug": unit[drug_col],
        "original_smiles": original,
        "canonical_smiles": cleaned,
    }).drop_duplicates()
    audit["valid_before"] = audit["original_smiles"].map(valid_smiles)
    audit["valid_after"] = audit["canonical_smiles"].map(valid_smiles)
    smiles_audit_parts.append(audit)

smiles_audit = pd.concat(smiles_audit_parts, ignore_index=True)

# Mark rows whose final structure came from a documented override.
smiles_audit["used_documented_override"] = smiles_audit["drug"].map(
    lambda x: normalize_drug_key(x) in SMILES_OVERRIDES
)
smiles_audit.to_csv(AUDIT_DIR / "smiles_correction_audit.csv", index=False)

rescued_drugs = sorted(
    smiles_audit.loc[smiles_audit["used_documented_override"], "drug"]
    .dropna().astype(str).unique().tolist()
)
print("Documented structure overrides applied to:", rescued_drugs)
print("Invalid role-specific structures before:", int((~smiles_audit["valid_before"]).sum()))
print("Invalid role-specific structures after:", int((~smiles_audit["valid_after"]).sum()))

if (~smiles_audit["valid_after"]).any():
    display(smiles_audit.loc[~smiles_audit["valid_after"]])
    raise ValueError("Invalid structures remain after documented cleaning.")


### Generate 512-bit Morgan fingerprints for both canonical drug roles


In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS,
    fpSize=MORGAN_BITS,
)

unique_smiles = pd.unique(
    pd.concat(
        [unit["drug_a_smiles"], unit["drug_b_smiles"]],
        ignore_index=True,
    )
)

fingerprint_cache = {}
for smiles in unique_smiles:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Unexpected invalid canonical SMILES: {smiles}")
    fingerprint = generator.GetFingerprint(mol)
    array = np.zeros((MORGAN_BITS,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fingerprint, array)
    fingerprint_cache[smiles] = array

for role in ["a", "b"]:
    matrix = np.vstack([
        fingerprint_cache[smiles]
        for smiles in unit[f"drug_{role}_smiles"]
    ])
    frame = pd.DataFrame(
        matrix,
        columns=[f"drug_{role}_fp_{i:04d}" for i in range(MORGAN_BITS)],
        index=unit.index,
        dtype=np.uint8,
    )
    unit = pd.concat([unit, frame], axis=1)

morgan_cols = sorted(
    [
        c for c in unit.columns
        if re.fullmatch(r"drug_[ab]_fp_\d{4}", str(c))
    ]
)

assert len(morgan_cols) == 2 * MORGAN_BITS
print("Morgan columns:", len(morgan_cols))


### Explicit feature policy: supervised versus descriptive metadata
The canonical dataset retains raw drug-specific annotation fields for traceability. The supervised model-family comparison selects and combines these into five symmetric pharmacological annotation features; it does not use all stored columns directly. Drug identities and clinical-relevance metadata are excluded from the predictive representation.

In [ ]:
identity_cols = [c for c in ["drug_a_norm", "drug_b_norm"] if c in unit.columns]

transferable_annotation_cols = [
    c for c in [
        "drug_a_target", "drug_b_target",
        "drug_a_pathway", "drug_b_pathway",
        "drug_a_drug_type", "drug_b_drug_type",
        "drug_a_drugcomb_target_name_raw",
        "drug_b_drugcomb_target_name_raw",
        "drug_a_drugcomb_target_type_raw",
        "drug_b_drugcomb_target_type_raw",
    ]
    if c in unit.columns
]

descriptive_only_cols = [
    c for c in [
        "drug_a_clinical_relevance",
        "drug_b_clinical_relevance",
        "drug_a_clinical_phase_drugcomb",
        "drug_b_clinical_phase_drugcomb",
    ]
    if c in unit.columns
]

feature_manifest_rows = []
for block, columns, supervised in [
    ("identity", identity_cols, False),
    ("transferable_annotations", transferable_annotation_cols, True),
    ("expression_pca", pca_cols, True),
    ("morgan_role_specific", morgan_cols, True),
    ("clinical_relevance_descriptive_only", descriptive_only_cols, False),
]:
    for column in columns:
        feature_manifest_rows.append({
            "feature_block": block,
            "column": column,
            "supervised_feature": supervised,
            "note": (
                "Retained as descriptive metadata; excluded from supervised modelling."
                if not supervised else ""
            ),
        })

feature_manifest = pd.DataFrame(feature_manifest_rows)
feature_manifest.to_csv(AUDIT_DIR / "canonical_feature_manifest.csv", index=False)

print("Identity columns:", len(identity_cols))
print("Transferable annotation columns:", len(transferable_annotation_cols))
print("Expression PCs:", len(pca_cols))
print("Morgan role-specific columns:", len(morgan_cols))
print("Descriptive-only clinical and phase metadata columns:",len(descriptive_only_cols))


### Hard-stop audit against the frozen thesis endpoint contract


In [ ]:
observed = {
    "concentration_rows": int(unit["n_rows"].sum()),
    "canonical_units": len(unit),
    "cell_lines": int(unit["ModelID"].nunique()),
    "unordered_drug_pairs": int(unit["candidate_pair_key"].nunique()),
    "any_positive_units": int(unit["any_positive_label"].sum()),
    "majority_positive_units": int(unit["majority_label"].sum()),
    "conflicting_units": int(unit["has_conflicting_labels"].sum()),
}

expected = {
    "concentration_rows": 53441,
    "canonical_units": 8450,
    "cell_lines": 26,
    "unordered_drug_pairs": 325,
    "any_positive_units": 1063,
    "majority_positive_units": 188,
    "conflicting_units": 999,
}

contract = pd.DataFrame([
    {
        "quantity": key,
        "expected": expected[key],
        "observed": observed[key],
        "pass": expected[key] == observed[key],
    }
    for key in expected
])
display(contract)

if not contract["pass"].all():
    raise AssertionError(
        "Canonical unit construction does not match the frozen thesis contract. "
        "Do not continue to modelling until the discrepancy is resolved."
    )

print("✓ Strict endpoint contract verified.")


### Save the canonical dataset and audit artifacts


In [ ]:
# Save a compact human-readable unit table without the 1,024 fingerprint columns.
compact_cols = [c for c in unit.columns if c not in morgan_cols]
unit[compact_cols].to_csv(AUTHORITATIVE_CSV, index=False)

# Full binary-feature table for downstream modelling.
unit.to_pickle(AUTHORITATIVE_PICKLE)

# Local-only published-notebook input: include all role-specific 512-bit Morgan
# features and all unit-level covariates. No drug- or cell-line rows are omitted.
unit.to_csv(CANONICAL_MODELLING_CSV, index=False)

# Full matched expression matrix for correct training-fold PCA in grouped
# unseen-cell-line validation: keep genes with limited missingness rather than
# reusing the globally-fitted descriptive PCA.
expression_for_fold_pca.reset_index().to_csv(EXPRESSION_FOLD_CSV, index=False)

# Read back structural contracts to catch accidental loss of identifiers.
_canonical_header = pd.read_csv(CANONICAL_MODELLING_CSV, nrows=0).columns
_expression_header = pd.read_csv(EXPRESSION_FOLD_CSV, nrows=0).columns
assert {"ModelID", "drug_a_norm", "drug_b_norm", "positive_fraction"}.issubset(_canonical_header)
assert "ModelID" in _expression_header and len(_expression_header) == 19216, (
    f"Expected 19,215 gene columns plus ModelID, "
    f"but found {len(_expression_header)} total columns."
)
assert len([c for c in _canonical_header if re.fullmatch(r"drug_[ab]_fp_\d{4}", c)]) == 1024

if WRITE_LEGACY_COMPATIBILITY_COPY:
    LEGACY_COMPAT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(AUTHORITATIVE_PICKLE, LEGACY_COMPAT_PATH)

output_paths = {
    "strict canonical unit CSV": AUTHORITATIVE_CSV,
    "canonical unit + Morgan pickle": AUTHORITATIVE_PICKLE,
    "local canonical Morgan modelling CSV": CANONICAL_MODELLING_CSV,
    "local 26-line full-expression CSV": EXPRESSION_FOLD_CSV,
}
if WRITE_LEGACY_COMPATIBILITY_COPY:
    output_paths["compatibility copy for existing modelling notebooks"] = LEGACY_COMPAT_PATH

output_manifest = pd.DataFrame([
    {
        "artifact": name,
        "path_relative_to_repo": rel(path),
        "size_mb": Path(path).stat().st_size / (1024 ** 2),
        "sha256": sha256_file(Path(path)),
    }
    for name, path in output_paths.items()
])
output_manifest.to_csv(AUDIT_DIR / "canonical_unit_output_manifest.csv", index=False)
display(output_manifest)

dataset_card = f"""# Canonical multi-cell-line PDAC unit dataset

## Biological unit
One row = one `ModelID × unordered drug-pair` unit.

## Final endpoint contract
- any-positive: `positive_fraction > 0`
- strict majority-positive: `positive_fraction > 0.5`

## Frozen counts
- concentration rows represented: {int(unit['n_rows'].sum()):,}
- canonical units: {len(unit):,}
- cell lines: {unit['ModelID'].nunique():,}
- unordered drug pairs: {unit['candidate_pair_key'].nunique():,}
- any-positive units: {int(unit['any_positive_label'].sum()):,}
- majority-positive units: {int(unit['majority_label'].sum()):,}
- conflicting-label units: {int(unit['has_conflicting_labels'].sum()):,}
- exact-50% units: {int(unit['exact_half_positive'].sum()):,}

## Chemical representation
Each canonical drug role has a 512-bit radius-2 Morgan fingerprint.
Downstream symmetric representations are constructed from the mean and absolute
difference of the two role-specific fingerprints.

## Feature policy
Clinical-relevance text is retained for descriptive traceability but explicitly excluded
from the supervised annotation block.
"""
(AUDIT_DIR / "canonical_unit_dataset_card.md").write_text(dataset_card, encoding="utf-8")
print(dataset_card)


## Final output contract and downstream use

| Artifact | Local path | Intended use |
| --- | --- | --- |
| Canonical Morgan-enriched unit table | `tables/pdac_multicell_drug_pair_units_with_morgan.csv` | `multi_cell_line_grouped_validation.ipynb`, `drug_pair_candidate_ranking.ipynb` |
| DepMap full expression matrix | `tables/depmap_pdac_full_expression_for_fold_pca.csv` | Training-only gene filtering, scaling and PCA in grouped unseen-cell-line validation |
| Canonical unit pickle | `data/processed/dataset_assembly/canonical_order_invariant_unit_with_morgan.pkl` | Optional local analysis and internal consistency checks |
| Concentration-level table | `data/processed/dataset_assembly/final_ml_ready_jaaks_pdac_enriched_with_expression_pca.csv` | Upstream audit and reconstruction |
| Audit tables and dataset cards | `results/tables/` | Reproducibility records (SHA-256, software, endpoint and structure audits) |

**Validation targets:** 53,441 concentration-level rows (2,483 positive), 8,450 units, 26 cell lines, 325 unordered drug pairs, 1,063 any-positive units, 188 strict majority-positive units, 999 mixed-label units, 206 exactly-half-positive units; 1,024 raw Morgan columns; expression matrix 26 × 19,215 features (19,216 CSV columns including ModelID). The original source notebook reported 19,215 *complete-case* genes used for its descriptive global PCA (20 components).

**Do not commit** the two large table CSVs, raw input files, or the generated `data/processed/dataset_assembly/` directory to public GitHub. The repository `.gitignore` already excludes the two `tables/` CSVs and `data/raw/`; also ignore `/data/processed/dataset_assembly/` before a broad `git add`.

This notebook reproduces the **author's original assembly logic**, including full-population PCA for descriptive within-cell-line features. For analyses holding out entire cell lines, use the separate full-expression matrix and refit PCA strictly within each training fold. Avoid treating descriptive global PCs as unseen-cell-line validation features.

For external experimental validation, its configuration cell may need `TRAINING_UNITS_PATH = "tables/pdac_multicell_drug_pair_units_with_morgan.csv"`; the notebook's existing default points to a historical local output location.